In [ ]:
%%bash

# install required system dependencies
apt-get install -y xvfb x11-utils

# install required python dependencies (might need to install additional gym extras depending)
pip install -q pyvirtualdisplay PyOpenGL PyOpenGL-accelerate gymnasium

In [ ]:
import base64
import glob
import io

import gymnasium as gym
import matplotlib.pyplot  as plt
import numpy as np
from gymnasium.wrappers.record_video import RecordVideo
from IPython.display import HTML
from IPython import display as ipythondisplay
from tqdm.notebook import tqdm

In [ ]:
VIDEO_FOLDER = './video/'

In [ ]:
def show_video():
    mp4list = glob.glob(f'{VIDEO_FOLDER}/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        ipythondisplay.display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")


def wrap_env(env):
    env = RecordVideo(env, VIDEO_FOLDER,
                      episode_trigger=lambda episode_number: True)
    return env


In [ ]:
env = wrap_env(gym.make("Taxi-v3", render_mode="rgb_array"))

def base_simulation(env):
    observation = env.reset()

    while True:

        env.render()

        action = env.action_space.sample()

        observation, reward, done, truncated, info = env.step(action)

        if done or truncated:
            break

    env.close()

    show_video()

base_simulation(env)

In [ ]:
pseudo código
receba o ambienete env

Q <- crie uma matriz de zeros (numero de estados vs numero de ações)
epsilon <- 1

repita n_epsodes vezes
    estado <- reinicia o ambiente

    repita n_steps vezes
       roleta <- receba um número aleatório entre 0, 1
       
        se roleta <- epsilon
           action <- ação aleatória

        senão
            action <- argmax(Q[estado])

        # aplica a ação no ambiente e receba o novo estado e uma recompensa
        novo_estado, reward, done <- env.step(action)

        # atualiza matriz Q na posição estado, action
        Q[estado, action] <-((1 - alpha) * Q[estado, action] +
                                  alpha  * (reward + gamma * max(Q[novo_estado])))

        se done:
            interrompa o loop

    # reduz o epsilon
    epsilon <- min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay * epoca_atual)

In [ ]:
env = wrap_env(gym.make("Taxi-v3", render_mode="rgb_array"))


def simulate_using_Q(env, Q, max_steps=100, debug=False):
    observation, _ = env.reset()

    for _ in range(max_steps):

        env.render()

        action = np.argmax(Q[observation,:])
        if debug:
            print(Q[observation,:])

        observation, reward, done, truncated, info = env.step(action)

        if done or truncated:
            break

    env.close()

    show_video()


# simulate_using_Q(env, Q)

In [ ]:
!pip install -q flappy-bird-gymnasium

In [ ]:
import random
from itertools import cycle
from typing import Tuple, Optional

import gymnasium
import pygame
from flappy_bird_gymnasium import FlappyBirdEnv
from flappy_bird_gymnasium.envs.constants import (
    PLAYER_HEIGHT, PLAYER_WIDTH, PIPE_HEIGHT, PIPE_WIDTH, PLAYER_MAX_VEL_Y
)


# quantization = [-.8,-.5, -.3, -.2, -.1, -.05, -0.02, 0.02, .05, .1, .2, .3, .5, .8]
quantizations = {
    'v_dist': [-.15, -.05, 0, .05, .15, .3],
    'player_y': [.2, .6],
    'vel_y': [-.5, -.1, .2, .5],
    'h_dist': [.2]
}


def bound(val, lower=-1, upper=1):
    return max(min(val, upper), lower)

class FlappyBirdDiscreteEnv(FlappyBirdEnv):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 30}

    def __init__(
        self,
        screen_size: Tuple[int, int] = (288, 512),
        pipe_gap: int = 100,
        bird_color: str = "yellow",
        pipe_color: str = "green",
        render_mode: Optional[str] = None,
        background: Optional[str] = "day",
        score_limit: Optional[int] = None,
        debug: bool = False,
    ) -> None:
        super().__init__(screen_size, False, True, False,
                         pipe_gap, bird_color, pipe_color, render_mode, background, score_limit, debug)

        self.shape = (10, 20, 20)
        self.observation_space = gymnasium.spaces.Discrete(7 * 2)


    def _get_observation_features_raw(self):
        up_pipe = low_pipe = None
        h_dist = 0
        prev_dist = -100
        pipes = sorted(zip(self._upper_pipes, self._lower_pipes),
                       key=lambda x: x[0]["x"])

        for up_pipe, low_pipe in pipes:
            h_dist = (low_pipe["x"] + PIPE_WIDTH / 2
                      - (self._player_x - PLAYER_WIDTH / 2))
            h_dist += 5  # extra distance to compensate for the buggy hit-box
            if h_dist >= 0:
                break

            prev_dist = h_dist

        upper_pipe_y = up_pipe["y"] + PIPE_HEIGHT
        lower_pipe_y = low_pipe["y"]
        player_y = self._player_y
        vel_y = self._player_vel_y

        v_dist = (upper_pipe_y + lower_pipe_y) / 2 - (player_y
                                                      + PLAYER_HEIGHT/2)

        h_dist /= self._screen_width
        prev_dist /= self._screen_width

        v_dist /= self._screen_height
        vel_y /= PLAYER_MAX_VEL_Y
        lower_pipe_y /= self._screen_height
        upper_pipe_y /= self._screen_height
        player_y /= self._screen_height

        return prev_dist, h_dist, v_dist, vel_y, lower_pipe_y, upper_pipe_y, player_y

    def _get_observation_features(self):
        (prev_dist, h_dist, v_dist, vel_y,
        lower_pipe_y, upper_pipe_y, player_y) = self._get_observation_features_raw()

        v_dist = np.digitize(v_dist, bins=quantizations['v_dist'])  % 8
        player_y = np.digitize(player_y, bins=quantizations['player_y'])  % 4
        vel_y = np.digitize(vel_y, bins=quantizations['vel_y'])  % 5
        h_dist = np.digitize(h_dist, bins=quantizations['h_dist'])  % 3
        prev_dist = np.digitize(-prev_dist, bins=quantizations['h_dist'])  % 3

        factor = 1
        state = (
            v_dist +
            # player_y * (factor := factor * (len(quantizations['v_dist']) + 1)) +
            # vel_y *    (factor := factor * (len(quantizations['v_dist']) + 1)) +
            h_dist * (factor := factor * (len(quantizations['v_dist']) + 1))
            # 0 if player_y <= lower_pipe_y else (1 if lower_pipe_y <= player_y <= upper_pipe_y else 2)
            # int(bound(h_dist, 0) * 10) +
            # int((bound(v_dist, -0.5, 0.5) + 0.5) * 20) * self.shape[0]
            # int((bound(vel_y) + 1) * 10 * self.shape[0] * self.shape[1])
        )

        return state, None

    def reset(self, seed=None, options=None):
        """Resets the environment (starts a new game)."""
        super().reset(seed=seed)

        self._player_x = int(self._screen_width * .2)
        self._player_y = int((self._screen_height - PLAYER_HEIGHT) / 2)
        # self._player_y = int(self._screen_height * np.random.normal(0.5, 0.1))


        obs, _ = self._get_observation()
        info = {"score": self._score}
        return obs, info

    def step(self, action):
        observation, reward, done, truncated, info  = super().step(action)
        (prev_dist, h_dist, v_dist, vel_y,
        lower_pipe_y, upper_pipe_y, player_y) = self._get_observation_features_raw()


        if 0.05 <= v_dist <= 0.15:
            reward = 0
        elif 0.15 <= v_dist:
            reward = -0.05
        elif 0.3 <= v_dist:
            reward = -0.1

        if action == 1 and vel_y < .1:
            reward =- 0.1

        # elif player_y <= 0.1 or player_y >= 0.6:
        #     reward = 0

        dists.append((v_dist, player_y, vel_y, h_dist))
        return observation, reward, done, truncated, info


dists  = []

In [ ]:
env = FlappyBirdDiscreteEnv(render_mode="rgb_array")

# chame sua função de aprendizado